In [2]:
%pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 618.0 kB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 291.5 kB/s  0:00:14 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [huggingface_hub] [huggingface_hub]
Note: you may need to restart the kernel to use updated packages.


Dataset **MedFact-Synth** dari Hugging Face berukuran besar (17 file Parquet, total ±3 GB), jadi saya tidak memakai cara sederhana seperti `pl.read_parquet("hf://...")` langsung. saya mengunduh dataset dengan empat langkah berikut:

1. **Unduh ke disk lokal (`snapshot_download`), bukan streaming langsung.**
   Membaca dataset lewat `hf://` langsung ke memori (`pl.read_parquet`) sering gagal atau lambat pada dataset besar, dan setiap restart kernel memaksa saya mengunduh ulang dari awal. saya menyimpan file `.parquet` ke folder `data/raw/` lebih dulu. Selama saya me-mount folder proyek ke container Docker, data tetap tersimpan dan notebook tidak perlu mengunduh ulang setiap kali berjalan.

2. **Unduh per batch (`BATCH_SIZE = 1`) dengan `max_workers = 1`.**
   Mengunduh 17 file secara paralel membebani RAM dan bandwidth, dan mempersulit pelacakan error: kalau satu file gagal, saya harus mengulang seluruh batch besar. Batch kecil memudahkan saya memantau progres per file, dan koneksi yang putus hanya merugikan satu file.

3. **Cek file yang sudah ada sebelum mengunduh (logika resume/skip).**
   Sebelum mengunduh, notebook memeriksa file apa saja yang sudah ada di `data/raw/` beserta ukurannya. Ukuran `> 0` byte menandakan file tidak korup dan sudah selesai, sehingga notebook melewati file tersebut. Karena itu prosesnya idempotent: kalau notebook restart atau container mati di tengah jalan, notebook melanjutkan dari file yang tersisa tanpa mengulang dari nol.

4. **Verifikasi akhir (jumlah file dan total ukuran).**
   Setelah semua batch selesai, notebook menghitung ulang jumlah file valid dan total ukurannya. Kami memastikan ke-17 file sudah lengkap sebelum masuk ke tahap profiling berikutnya.

In [1]:
from huggingface_hub import snapshot_download, list_repo_files
from pathlib import Path
import time

REPO_ID = "Xingchi888/MedFact-Synth"
DATA_PATH = Path("/home/jovyan/work/data/raw/")
BATCH_SIZE = 1
MAX_WORKERS = 1

DATA_PATH.mkdir(parents=True, exist_ok=True)

print("MENGAMBIL DAFTAR FILE")

all_files = list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

parquet_files = sorted([
    f
    for f in all_files
    if f.startswith("data/train-")
    and f.endswith(".parquet")
])

print(f"Total file Parquet : {len(parquet_files)}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Lokasi download    : {DATA_PATH.resolve()}")
print()

downloaded_files = list(DATA_PATH.rglob("*.parquet"))

existing_files = {
    file.name
    for file in downloaded_files
    if file.is_file() and file.stat().st_size > 0
}

print("VERIFIKASI FILE")
print(f"File sudah ada : {len(existing_files)}")
print(f"File belum ada : {len(parquet_files) - len(existing_files)}")
print()

if existing_files:
    print("File yang akan dilewati:")

    for file in sorted(existing_files):
        print(f"  {file}")

    print()

remaining_files = [
    file
    for file in parquet_files
    if Path(file).name not in existing_files
]

print("FILE YANG PERLU DIDOWNLOAD")

for file in remaining_files:
    print(f"  {file}")

print()

if not remaining_files:
    print("Semua file sudah tersedia")

else:
    total_batches = (
        len(remaining_files) + BATCH_SIZE - 1
    ) // BATCH_SIZE

    print(f"Total batch yang diperlukan: {total_batches}")
    print()

    for i in range(
        0,
        len(remaining_files),
        BATCH_SIZE
    ):

        batch_number = (
            i // BATCH_SIZE
        ) + 1

        batch = remaining_files[
            i:i + BATCH_SIZE
        ]

        print(f"BATCH {batch_number}/{total_batches}")

        for file in batch:
            print(f"  {file}")

        start_time = time.time()

        try:

            snapshot_download(
                repo_id=REPO_ID,
                repo_type="dataset",
                allow_patterns=batch,
                local_dir=str(DATA_PATH),
                max_workers=MAX_WORKERS,
            )

            elapsed = time.time() - start_time

            print()
            print("BATCH SELESAI")
            print(f"Waktu : {elapsed / 60:.2f} menit")
            print()

        except Exception as e:

            print()
            print("BATCH GAGAL")
            print(f"Error: {e}")
            print()
            print("Program dihentikan.")
            print("Jalankan kembali cell untuk melanjutkan.")

            raise

print()
print("VERIFIKASI AKHIR")

downloaded_files = sorted(
    DATA_PATH.rglob("*.parquet")
)

valid_files = [
    file
    for file in downloaded_files
    if file.is_file()
    and file.stat().st_size > 0
]

print(f"File Parquet ditemukan : {len(valid_files)}")
print(f"Lokasi : {DATA_PATH.resolve()}")

total_size = sum(
    file.stat().st_size
    for file in valid_files
)

print(
    f"Total ukuran : "
    f"{total_size / (1024**3):.2f} GB"
)

print()
print("FILE YANG TERSEDIA")

for file in valid_files:

    size_mb = (
        file.stat().st_size
        / (1024**2)
    )

    print(
        f"{file.name:<40} "
        f"{size_mb:,.2f} MB"
    )

print()

if len(valid_files) == len(parquet_files):
    print("Semua file berhasil didownload")
else:
    print(
        f"Masih ada "
        f"{len(parquet_files) - len(valid_files)} "
        f"file yang belum tersedia"
    )

MENGAMBIL DAFTAR FILE
Total file Parquet : 17
Batch size         : 1
Lokasi download    : /home/jovyan/work/data/raw

VERIFIKASI FILE
File sudah ada : 16
File belum ada : 1

File yang akan dilewati:
  train-00000-of-00017.parquet
  train-00001-of-00017.parquet
  train-00002-of-00017.parquet
  train-00003-of-00017.parquet
  train-00004-of-00017.parquet
  train-00005-of-00017.parquet
  train-00006-of-00017.parquet
  train-00007-of-00017.parquet
  train-00008-of-00017.parquet
  train-00009-of-00017.parquet
  train-00010-of-00017.parquet
  train-00011-of-00017.parquet
  train-00012-of-00017.parquet
  train-00013-of-00017.parquet
  train-00014-of-00017.parquet
  train-00015-of-00017.parquet

FILE YANG PERLU DIDOWNLOAD
  data/train-00016-of-00017.parquet

Total batch yang diperlukan: 1

BATCH 1/1
  data/train-00016-of-00017.parquet


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]


BATCH SELESAI
Waktu : 0.91 menit


VERIFIKASI AKHIR
File Parquet ditemukan : 17
Lokasi : /home/jovyan/work/data/raw
Total ukuran : 3.07 GB

FILE YANG TERSEDIA
train-00000-of-00017.parquet             182.82 MB
train-00001-of-00017.parquet             183.30 MB
train-00002-of-00017.parquet             184.93 MB
train-00003-of-00017.parquet             186.10 MB
train-00004-of-00017.parquet             185.92 MB
train-00005-of-00017.parquet             185.75 MB
train-00006-of-00017.parquet             185.36 MB
train-00007-of-00017.parquet             185.41 MB
train-00008-of-00017.parquet             185.61 MB
train-00009-of-00017.parquet             185.38 MB
train-00010-of-00017.parquet             183.53 MB
train-00011-of-00017.parquet             184.26 MB
train-00012-of-00017.parquet             186.36 MB
train-00013-of-00017.parquet             186.07 MB
train-00014-of-00017.parquet             185.74 MB
train-00015-of-00017.parquet             185.41 MB
train-00016-of-00017.par

## Data Profiling

Urutan pemeriksaan:
1. Ukuran data dan skema kolom
2. Menampilkan 10 Data teratas


In [1]:
import polars as pl

DATA_GLOB = "/home/jovyan/work/data/raw/data/*.parquet"

lf = pl.scan_parquet(DATA_GLOB)
schema = lf.collect_schema()

n_rows = lf.select(pl.len()).collect().item()
n_cols = len(schema)

print(f"Jumlah baris : {n_rows:,}")
print(f"Jumlah kolom : {n_cols}")
print()
print("Skema kolom:")
for name, dtype in schema.items():
    print(f"  {name:<30} {dtype}")


Jumlah baris : 1,497,981
Jumlah kolom : 10

Skema kolom:
  idx                            Int64
  claim_pmid                     String
  claim_potential                String
  claim                          String
  source_pmid                    String
  source                         String
  synthetic_label                Float64
  system_prompt                  String
  user_prompt                    String
  assistant_prompt               String


In [2]:
lf.head(10).collect()


idx,claim_pmid,claim_potential,claim,source_pmid,source,synthetic_label,system_prompt,user_prompt,assistant_prompt
i64,str,str,str,str,str,f64,str,str,str
0,"""30588127""","""support""","""An educational intervention us…","""30588127""","""An educational intervention ba…",2.0,"""You are a fact-checking expert…","""Article: An educational interv…","""<think>The article discusses a…"
1,"""30588127""","""support""","""An educational intervention us…","""24523424""","""A systematic review of barrier…",0.0,"""You are a fact-checking expert…","""Article: A systematic review o…","""<think>The article discusses b…"
2,"""30588127""","""support""","""An educational intervention us…","""30604386""","""The Effect of Educational Inte…",-1.0,"""You are a fact-checking expert…","""Article: The Effect of Educati…","""<think>The article discusses a…"
3,"""34346072""","""support""","""Older cancer survivors experie…","""34346072""","""Functional decline among older…",2.0,"""You are a fact-checking expert…","""Article: Functional decline am…","""<think>The article directly in…"
4,"""11255101""","""support""","""A strong humoral immune respon…","""11497076""","""Relationships between the pres…",2.0,"""You are a fact-checking expert…","""Article: Relationships between…","""<think>The article states that…"
5,"""11255101""","""support""","""A strong humoral immune respon…","""11255101""","""Antibodies against full-length…",2.0,"""You are a fact-checking expert…","""Article: Antibodies against fu…","""<think>The article directly in…"
6,"""11255101""","""support""","""A strong humoral immune respon…","""1723280""","""Low number of functionally act…",0.0,"""You are a fact-checking expert…","""Article: Low number of functio…","""<think>The article discusses t…"
7,"""30092""","""support""","""Kainate inhibits synaptic tran…","""30092""","""Kainic acid and synaptic trans…",2.0,"""You are a fact-checking expert…","""Article: Kainic acid and synap…","""<think>The article discusses t…"
8,"""30092""","""support""","""Kainate inhibits synaptic tran…","""2863795""","""Evidence for L-glutamate as th…",-1.0,"""You are a fact-checking expert…","""Article: Evidence for L-glutam…","""<think>The article discusses t…"
